# CPG Beverage Sentiment Analysis - Large-Scale Fine-Tuning in Google Colab

This notebook enables you to run the complete, production-grade sentiment modeling pipeline on the full **100,000+ reviews** using Google Colab's free **T4 GPU** hardware acceleration. This will achieve the **90–92% F1 score** targets.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/datasciharshit/CPG_Beer_Reviews_Sentiment_Analysis/blob/main/CPG_Beer_Reviews_Fine_Tuning.ipynb)

## Step 1: Clone Repository and Install Dependencies
We will clone the Github repository and install the custom `beverage_cleaner` package in editable mode.

In [ ]:
# Clone the repository
!git clone https://github.com/datasciharshit/CPG_Beer_Reviews_Sentiment_Analysis.git
%cd CPG_Beer_Reviews_Sentiment_Analysis

# Install dependencies and package
!pip install -r requirements.txt
!pip install -e .

## Step 2: Mount Google Drive to Load the Dataset
Because the raw dataset file (`beer_reviews_clean_100k.csv`) is larger than GitHub's file limits and ignored by `.gitignore`, you should upload it to your Google Drive, then mount it in Colab to copy it into the working folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# TODO: Update this path to where you uploaded 'beer_reviews_clean_100k.csv' in your Google Drive
# Example: '/content/drive/MyDrive/Colab Notebooks/beer_reviews_clean_100k.csv'
gdrive_dataset_path = '/content/drive/MyDrive/beer_reviews_clean_100k.csv'

import os
os.makedirs('dataset', exist_ok=True)

if os.path.exists(gdrive_dataset_path):
    !cp "$gdrive_dataset_path" dataset/beer_reviews_clean_100k.csv
    print("✅ Dataset successfully copied to local environment.")
else:
    print("❌ Dataset not found at the specified Google Drive path. Please upload and update the path above.")

## Step 3: Run Full Parallel Text Preprocessing
We will clean and structure the full dataset. Since Google Colab has multiple virtual CPU cores, this parallel step will run highly optimized.

In [ ]:
# Let's modify the script parameters dynamically to run on the full 100K reviews
import pathlib
script_path = pathlib.Path('run_dataset_test.py')
code = script_path.read_text()

# Remove the N_ROWS cap so it runs on all 100,000+ reviews
code = code.replace("df = pd.read_csv(DATASET_PATH, nrows=N_ROWS)", "df = pd.read_csv(DATASET_PATH)")
script_path.write_text(code)

# Run the processing pipeline
!python run_dataset_test.py

## Step 4: Run Production Model Fine-Tuning
Now we will execute the model training pipeline in full production mode (using the GPU accelerator). 
We will set it to train on **20,000 training reviews** and evaluate on **2,000 validation reviews** for **3 epochs** to converge at the target **90–92% F1 score**.

In [ ]:
# Set production parameters in modeling configurations
import pathlib
script_path = pathlib.Path('run_modeling.py')
code = script_path.read_text()
code = code.replace("fine_tune_train_size = 5000", "fine_tune_train_size = 20000")
code = code.replace("fine_tune_eval_size = 1000", "fine_tune_eval_size = 2000")
code = code.replace("epochs = 1", "epochs = 3")
script_path.write_text(code)

# Run model training and evaluations with CUDA GPU acceleration
!python run_modeling.py --fine-tune --no-quick-run

## Step 5: Save and Zip Trained Checkpoints
Zip the resulting fine-tuned weights and model checkpoints so you can download them or copy them directly back to your Google Drive to be loaded locally by the Streamlit dashboard.

In [ ]:
# Zip the artifacts folder
!zip -r fine_tuned_artifacts.zip artifacts/

# Copy back to Google Drive
!cp fine_tuned_artifacts.zip "/content/drive/MyDrive/"
print("✅ Fine-tuned artifacts zipped and saved to your Google Drive!")